# Phase 4 Continued Training Notebook

This notebook continues training from the current Phase 3 PPO model into Phase 4 using the existing training pipeline in scripts/train_phase4.py.

Run cells top to bottom.

In [ ]:
from pathlib import Path
import subprocess
import os
import sys

ROOT = Path.cwd()
print(f'Workspace: {ROOT}')

resume_model = ROOT / 'models' / 'ppo_phase3_showdown_new' / 'best_model.zip'
train_script = ROOT / 'scripts' / 'train_phase4.py'

print(f'Resume model exists: {resume_model.exists()} -> {resume_model}')
print(f'Train script exists: {train_script.exists()} -> {train_script}')

if not resume_model.exists():
    raise FileNotFoundError(f'Missing model: {resume_model}')
if not train_script.exists():
    raise FileNotFoundError(f'Missing script: {train_script}')

In [ ]:
# Training configuration
ALGORITHM = 'PPO'
BACKEND = 'showdown'
FORMAT = 'gen9randombattle'

TIMESTEPS = 500_000
N_ENVS = 8
CHECKPOINT_FREQ = 50_000
EVAL_FREQ = 25_000

SAVE_PATH = 'models/ppo_phase4_from_phase3'
LOG_DIR = 'logs/phase4_from_phase3'

# Resume path should be passed without .zip to match script behavior
RESUME = 'models/ppo_phase3_showdown_new/best_model'

print('Configuration loaded')
print({
    'algorithm': ALGORITHM,
    'backend': BACKEND,
    'format': FORMAT,
    'timesteps': TIMESTEPS,
    'n_envs': N_ENVS,
    'checkpoint_freq': CHECKPOINT_FREQ,
    'eval_freq': EVAL_FREQ,
    'save_path': SAVE_PATH,
    'log_dir': LOG_DIR,
    'resume': RESUME,
})

In [ ]:
# Build training command
cmd = [
    sys.executable,
    'scripts/train_phase4.py',
    '--algorithm', ALGORITHM,
    '--backend', BACKEND,
    '--format', FORMAT,
    '--timesteps', str(TIMESTEPS),
    '--n-envs', str(N_ENVS),
    '--checkpoint-freq', str(CHECKPOINT_FREQ),
    '--eval-freq', str(EVAL_FREQ),
    '--save-path', SAVE_PATH,
    '--log-dir', LOG_DIR,
    '--resume', RESUME,
]

print('Command:')
print(' '.join(cmd))

In [ ]:
# Run Phase 4 continued training
# This can take a long time depending on TIMESTEPS and N_ENVS
env = os.environ.copy()
env['PYTHONPATH'] = str(ROOT)

result = subprocess.run(cmd, cwd=ROOT, env=env, check=False)
print(f'Exit code: {result.returncode}')
if result.returncode != 0:
    raise RuntimeError('Training failed. Check output above for details.')

## Optional: Evaluate the newly trained model

After training completes, run the next cell to evaluate the final model.

In [ ]:
final_model = f'{SAVE_PATH}/final_model'
eval_cmd = [
    sys.executable,
    'scripts/evaluate_rl.py',
    '--model', final_model,
    '--algorithm', ALGORITHM,
    '--episodes', '50',
    '--opponent', 'heuristic',
]

print('Eval command:')
print(' '.join(eval_cmd))

env = os.environ.copy()
env['PYTHONPATH'] = str(ROOT)
subprocess.run(eval_cmd, cwd=ROOT, env=env, check=False)